In [ ]:
!pip -q install transformers accelerate
import json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

train = pd.read_csv("train_comments.csv")
val = pd.read_csv("validation_comments.csv")
cats = lambda s: {a["category"] for a in json.loads(s)}

both = pd.concat([train, val])
labels = sorted(set().union(*[cats(s) for s in both.aspects]))
lab2id = {c: i for i, c in enumerate(labels)}
valid = {f: sorted(set().union(*[cats(s) for s in g.aspects])) for f, g in both.groupby("facility")}

NAME = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(NAME)

class DS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok([f"{f.replace('_', ' ')}: {t}" for f, t in zip(df.facility, df.text)],
                       truncation=True, max_length=96, padding=True)
        self.y = np.zeros((len(df), len(labels)), dtype="float32")
        for i, s in enumerate(df.aspects):
            for c in cats(s):
                self.y[i, lab2id[c]] = 1
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        d["labels"] = torch.tensor(self.y[i])
        return d

model = AutoModelForSequenceClassification.from_pretrained(
    NAME, num_labels=len(labels), problem_type="multi_label_classification")
args = TrainingArguments(output_dir="out", num_train_epochs=20, per_device_train_batch_size=16,
                         learning_rate=5e-5, weight_decay=0.01, save_strategy="no",
                         logging_steps=100, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=DS(train))
trainer.train()

def probs(df):
    return 1 / (1 + np.exp(-trainer.predict(DS(df)).predictions))

def score(df, p, thr):
    tp = fp = fn = 0
    for i, (f, s) in enumerate(zip(df.facility, df.aspects)):
        gold = cats(s)
        pred = {labels[j] for j in range(len(labels)) if p[i, j] >= thr and labels[j] in valid[f]}
        tp += len(gold & pred); fp += len(pred - gold); fn += len(gold - pred)
    P = tp / (tp + fp) if tp + fp else 0
    R = tp / (tp + fn)
    return P, R, (2 * P * R / (P + R) if P + R else 0)

pv = probs(val)
print("\nVALIDATION (keyword baseline: P 0.733  R 0.667  F1 0.698)")
for thr in (0.1, 0.2, 0.3, 0.4, 0.5):
    P, R, F = score(val, pv, thr)
    print(f"  confidence >= {thr}: precision={P:.3f} recall={R:.3f} F1={F:.3f}")

trainer.save_model("aspect_distilbert_v1")
tok.save_pretrained("aspect_distilbert_v1")
json.dump({"labels": labels, "valid": valid}, open("aspect_distilbert_v1/labels.json", "w"))
!zip -qr aspect_distilbert_v1.zip aspect_distilbert_v1
from google.colab import files
files.download("aspect_distilbert_v1.zip")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

...

Step,Training Loss
100,0.254783
200,0.122621
300,0.105399
400,0.082716
500,0.063382
600,0.051756
700,0.046060



VALIDATION (keyword baseline: P 0.733  R 0.667  F1 0.698)
  confidence >= 0.1: precision=0.430 recall=0.795 F1=0.559
  confidence >= 0.2: precision=0.686 recall=0.629 F1=0.656
  confidence >= 0.3: precision=0.772 recall=0.462 F1=0.578
  confidence >= 0.4: precision=0.869 recall=0.402 F1=0.549
  confidence >= 0.5: precision=0.932 recall=0.311 F1=0.466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>